# CartPole Training Notebook

In [ ]:

import gym
import numpy as np
import torch, torch.nn as nn, torch.optim as optim
import matplotlib.pyplot as plt

env = gym.make("CartPole-v1")

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(4, 128), nn.ReLU(),
            nn.Linear(128, 2)
        )
    def forward(self, x): return self.model(x)

net = Net()
opt = optim.Adam(net.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

rewards = []
gamma = 0.99

for episode in range(200):
    s,_ = env.reset()
    total = 0
    for t in range(500):
        s_t = torch.tensor(s, dtype=torch.float32)
        q = net(s_t)
        a = torch.argmax(q).item()
        s2, r, done, trunc, _ = env.step(a)
        total += r

        with torch.no_grad():
            target = q.clone()
            q2 = net(torch.tensor(s2, dtype=torch.float32))
            target[a] = r + gamma * torch.max(q2).item() * (1 - done)

        loss = loss_fn(q, target)
        opt.zero_grad(); loss.backward(); opt.step()

        s = s2
        if done: break
    rewards.append(total)

plt.plot(rewards); plt.title("CartPole Training Reward"); plt.show()

print("Training complete.")
